# 12 · Optimizar parametros con Optuna — nivel EXPERTO

**Que hace**: busca, con [Optuna](https://optuna.org), los parametros del motor
(umbral de score, similitud minima de nombre, distancia maxima de NIT…) que mejor
reproducen una **verdad etiquetada por personas**, y los aplica con
`linkage(ajustes_perfil=...)` para dejar la carpeta del estandar de salida.

**Que necesita**: una base de empresas (NIT, razon social y, opcionalmente, ciudad).
Si no tiene ninguna, el notebook genera una **base de demostracion con empresas
inventadas** cuya verdad se conoce (columna `ENTIDAD`), y con ella simula las
etiquetas que en la vida real pone una persona.

**Flujo (6 pasos, una celda cada uno)**:
1. Cargar la base (o generar la de demostracion).
2. Construir la **muestra de pares para etiquetar** con `GroundTruthGenerator`.
3. Etiquetar (`IS_MATCH`): simulado aqui; humano en la vida real.
4. Optimizar con `OrchestratorOptimizer` (pocos trials: ≤ 3 minutos).
5. Aplicar los mejores parametros con `linkage()` y comparar antes/despues.
6. Guardar el estudio como JSON.

Este notebook es **delgado a proposito**: configura y llama a la libreria; no
reimplementa ni el muestreo, ni la metrica, ni el bucle de optimizacion. Lo que
hace cada pieza y sus limites esta en [`docs/OPTIMIZACION.md`](../docs/OPTIMIZACION.md).

> En Colab Free, suba su archivo a `/content/datos` y descargue `/content/resultados`
> antes de cerrar la sesion; el disco local es efimero.
>
> Motor: `rues-linker 0.22.4` (`evaluation.OrchestratorOptimizer`, el UNICO
> optimizador de la libreria; `evaluation.GroundTruthGenerator`; `linkage`).
> Requiere las extras `optimization` (optuna) y `viz` (matplotlib).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_OBJETIVO = "0.22.4"
# Las extras traen optuna (optimization) y matplotlib (viz): sin ellas el
# optimizador y las figuras fallan con un ImportError que dice que instalar.
ORIGEN_GIT = (
    "rues-linker[optimization,viz] @ git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"
)
EXTRAS = ("optuna", "matplotlib")


def _version_ok(v: str, objetivo: str = VERSION_OBJETIVO) -> bool:
    """True si la version instalada coincide con el release objetivo."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) == _t(objetivo)
    except Exception:
        return False


def _faltan_extras() -> list[str]:
    """Las extras que no se pueden importar (optuna y matplotlib)."""
    return [m for m in EXTRAS if importlib.util.find_spec(m) is None]


def preparar_entorno() -> None:
    """Garantiza exactamente rues-linker 0.22.4 con sus extras desde el tag oficial."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada) and not _faltan_extras():
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} con optuna y matplotlib — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub con sus extras...")
    elif _faltan_extras():
        print(f"rues-linker {instalada} sin las extras {_faltan_extras()}; instalando...")
    else:
        print(f"rues-linker {instalada} no coincide con {VERSION_OBJETIVO}; instalando...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada. "
            "Reinicie el entorno y vuelva a ejecutar esta celda."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_OBJETIVO}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    if _faltan_extras():
        raise RuntimeError(
            f"Se instalo rues-linker pero faltan {_faltan_extras()}: "
            "ejecute `pip install optuna matplotlib` y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} con optuna y matplotlib — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS  (donde estan sus datos y donde quedan los resultados)
# ══════════════════════════════════════════════════════════════════════════
# En Colab Free suba sus archivos a /content/datos (este notebook no monta
# Drive) y descargue RUTA_RESULTADOS antes de cerrar la sesion: el disco local
# es efimero. RUTA_TRABAJO guarda los intermedios de linkage() (SQLite,
# checkpoints) y es disco LOCAL: nunca Drive. tests/test_notebooks_contrato.py
# sobreescribe estas rutas para correr el notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/datos")  # carpeta con sus archivos
RUTA_RESULTADOS = Path("/content/resultados/12")  # carpeta de salida (descarguela)
RUTA_TRABAJO = Path("/content/rues_linker_work/12")  # intermedios: disco LOCAL, nunca Drive

### Por que hace falta una verdad etiquetada

Ningun parametro es bueno o malo en abstracto: un umbral mas bajo une mas (sube el
recall y bajan la precision), uno mas alto une menos. La unica forma honesta de
elegir es medir contra pares que una persona reviso. Por eso el notebook construye
primero una **muestra estratificada** (pares faciles, dificiles, ambiguos y
negativos) con `GroundTruthGenerator`: una muestra al azar estaria llena de pares
triviales y no mediria nada donde el motor puede equivocarse.

Regla de la casa: **unir dos empresas distintas duele mas que dejar sin unir**. Si
su caso es asi, optimice `precision` o `f1`, no `recall`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from IPython.display import Image, display

from record_linkage import crear_config_orchestrator, linkage
from record_linkage.evaluation import (
    ConfigGroundTruth,
    GroundTruthGenerator,
    OrchestratorOptimizer,
    espacio_de_busqueda,
    graficar_convergencia,
    graficar_importancia,
    validar_etiquetas,
)
from record_linkage.matching.identificadores import bases_canonicas, digito_verificacion_dian

# ── Base de demostracion: empresas INVENTADAS con verdad conocida ─────────
# Nombres armados con palabras genericas (giro + lugar): ninguna empresa real.
GIROS = (
    "COMERCIALIZADORA", "DISTRIBUIDORA", "INVERSIONES", "TRANSPORTES", "INDUSTRIAS",
    "AGROPECUARIA", "FERRETERIA", "TEXTILES", "LOGISTICA", "CONSTRUCCIONES",
    "ALIMENTOS", "SERVICIOS",
)
LUGARES = (
    "DEL NORTE", "DEL SUR", "ANDINA", "DEL CARIBE", "LA MONTAÑA", "EL ROBLE",
    "RIO CLARO", "PUERTO AZUL", "VALLE VERDE", "MONTE ALTO", "LA PRADERA", "SAN LORENZO",
)
SUFIJOS = (("SAS", "S.A.S."), ("LTDA", "LIMITADA"), ("SA", "S.A."), ("", ""))
CIUDADES = ("BOGOTA", "MEDELLIN", "CALI", "BARRANQUILLA", "BUCARAMANGA")


def generar_base_demo(n_entidades: int = 120, semilla: int = 42) -> pd.DataFrame:
    """Base inventada con 1 a 4 registros por entidad y la verdad en ENTIDAD.

    Cada registro adicional trae una variacion de las que el motor encuentra en
    la vida real: (1) otro sufijo juridico (SAS vs S.A.S.), (2) un error de
    digitacion y el NIT con digito de verificacion, (3) sin sufijo y SIN NIT (el
    motor solo tiene el nombre). Como giro y lugar se combinan, hay entidades
    distintas con nombres parecidos (TRANSPORTES DEL NORTE / TRANSPORTES DEL SUR):
    los pares ambiguos existen de verdad.
    """
    rng = np.random.default_rng(semilla)
    nombres = np.array([f"{giro} {lugar}" for giro in GIROS for lugar in LUGARES])
    nombres = nombres[rng.permutation(len(nombres))[:n_entidades]]
    entidades = pd.DataFrame(
        {
            "ENTIDAD": [f"E{i:03d}" for i in range(n_entidades)],
            "NOMBRE": nombres,
            "NIT_BASE": (800_000_000 + 13 * np.arange(n_entidades)).astype(str),
            "SUFIJO": rng.integers(0, len(SUFIJOS), n_entidades),
            "CIUDAD": rng.choice(CIUDADES, n_entidades),
            "N_REGISTROS": rng.integers(1, 5, n_entidades),
        }
    )
    base = entidades.loc[entidades.index.repeat(entidades["N_REGISTROS"])].reset_index(drop=True)
    variante = base.groupby("ENTIDAD").cumcount().to_numpy()
    sufijo_a = np.array([s[0] for s in SUFIJOS])[base["SUFIJO"]]
    sufijo_b = np.array([s[1] for s in SUFIJOS])[base["SUFIJO"]]
    nombre = base["NOMBRE"]
    con_error = nombre.str.replace(r"^(\w)(\w)(\w)", r"\1\3\2", regex=True)  # CMOERCIAL...
    con_dv = base["NIT_BASE"] + "-" + base["NIT_BASE"].map(digito_verificacion_dian)
    razon_social = np.select(
        [variante == 1, variante == 2, variante == 3],
        [nombre + " " + sufijo_b, con_error + " " + sufijo_a, nombre],
        default=nombre + " " + sufijo_a,
    )
    nit = np.select([variante == 2, variante == 3], [con_dv, ""], default=base["NIT_BASE"])
    return pd.DataFrame(
        {
            "ID": [f"R{i:04d}" for i in range(len(base))],
            "NIT": nit,
            "RAZON_SOCIAL": pd.Series(razon_social).str.strip(),
            "CIUDAD": base["CIUDAD"].to_numpy(),
            "ENTIDAD": base["ENTIDAD"].to_numpy(),
        }
    )

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURACION               (LA UNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
ARCHIVO = "empresas_demo.csv"  # en RUTA_DATOS; si no existe se genera la base de demostracion
COL_ID = "ID"  # columna unica por fila, con el nombre que tenga (si no existe, se crea)
COL_NIT = "NIT"
COL_NOMBRE = "RAZON_SOCIAL"
COL_CIUDAD = "CIUDAD"  # None si no tiene
COL_ENTIDAD = "ENTIDAD"  # verdad conocida para SIMULAR etiquetas; None con datos reales
ARCHIVO_ETIQUETAS = "pares_etiquetados.xlsx"  # en RUTA_DATOS, solo si COL_ENTIDAD es None

PERFIL = "produccion_estandar"  # el perfil que linkage() usa por defecto
N_PARES = 160  # pares a etiquetar (20 % faciles, 30 % dificiles, 30 % ambiguos, 20 % negativos)
N_TRIALS = 8  # pocos: cada trial corre el motor completo sobre la base
OBJETIVO = "f1"  # f1 | f2 | precision | recall (macro_f1 exige verdad por registro con ESTRATO)
SEMILLA = 42
# Rango por parametro: (min, max, paso) o (min, max) para numeros, [opciones]
# para categoricos, un valor para dejarlo fijo. Solo claves que PERFIL declare:
# una que no este (p. ej. nit_empty_passes_filter en produccion_estandar) falla
# al construir el optimizador, porque linkage(ajustes_perfil=...) la rechazaria.
ESPACIO = {
    "score_threshold": (0.40, 0.80, 0.05),
    "min_name_similarity": (0.30, 0.80, 0.05),
    "lsh_threshold": (0.45, 0.70, 0.05),
    "max_nit_distance": (0, 3),
}

### 1 · La base

Se lee `ARCHIVO` desde `RUTA_DATOS` con todo como texto (un NIT leido como numero
pierde los ceros a la izquierda). Si el archivo no existe se genera la base de
demostracion y se guarda ahi mismo, para que el resto del notebook sea identico con
datos propios o inventados.

In [ ]:
RUTA_DATOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
ruta_base = RUTA_DATOS / ARCHIVO
if not ruta_base.exists():
    generar_base_demo(semilla=SEMILLA).to_csv(ruta_base, index=False)
    print(f"No habia {ARCHIVO}: se genero la base de demostracion (empresas inventadas).")

base = pd.read_csv(ruta_base, dtype=str).fillna("")
faltan = [c for c in (COL_NIT, COL_NOMBRE) if c not in base.columns]
if faltan:
    raise KeyError(
        f"A {ARCHIVO} le faltan las columnas {faltan}. Columnas disponibles: {list(base.columns)}. "
        "Corrija COL_NIT/COL_NOMBRE en la Celda B."
    )
if COL_ID not in base.columns:
    base[COL_ID] = [f"R{i:04d}" for i in range(len(base))]
if not base[COL_ID].is_unique:
    raise ValueError(f"{COL_ID} se repite: la verdad y la correlativa se cruzan por esa columna.")
# La base canonica del NIT (sin digito de verificacion) es la que decide si dos
# identificadores son el mismo; la calcula la libreria, no el notebook.
base["NIT_BASE"] = bases_canonicas(base[COL_NIT].to_numpy())

columnas_fuente = [COL_ID, COL_NIT, COL_NOMBRE] + ([COL_CIUDAD] if COL_CIUDAD else [])
FUENTES = {"BASE": base[columnas_fuente].copy()}
print(f"{len(base):,} registros · {base['NIT_BASE'].ne('').sum():,} con NIT")
if COL_ENTIDAD:
    print(f"{base[COL_ENTIDAD].nunique():,} entidades reales (verdad conocida en {COL_ENTIDAD})")
base.head(8)

### 2 · La muestra de pares para etiquetar

`GroundTruthGenerator` reparte `N_PARES` en cuatro clases: **faciles** (mismo
NIT), **dificiles** (nombre casi igual, NIT distinto o ausente), **ambiguos**
(nombre parecido a medias) y **negativos** (no se parecen). Los pares salen con
`IS_MATCH` vacio y se guardan en `pares_para_etiquetar.xlsx`: ese es el archivo que
una persona abre y llena. Los ids de cada par (`ID_1`, `ID_2`) son los de `COL_ID`.

In [ ]:
generador = GroundTruthGenerator(
    ConfigGroundTruth(
        semilla=SEMILLA,
        col_nombre=COL_NOMBRE,  # el notebook no preprocesa: se compara la razon social tal cual
        col_nit_canonico="NIT_BASE",
        col_nit=COL_NIT,
        col_razon_social=COL_NOMBRE,
    )
)
pares = generador.create_intelligent_sample(
    base.set_index(COL_ID),  # el indice es el id que llevan ID_1 / ID_2
    n_samples=N_PARES,
    output_path=RUTA_RESULTADOS / "pares_para_etiquetar.xlsx",
)
print(pares["CASE_TYPE"].value_counts().rename("pares por clase"))
pares[["ID_1", "ID_2", "RAZON_SOCIAL_1", "RAZON_SOCIAL_2", "NIT_1", "NIT_2", "NAME_SIMILARITY", "CASE_TYPE"]].head(8)

### 3 · Etiquetar

**En la vida real** una persona abre `pares_para_etiquetar.xlsx` y escribe en
`IS_MATCH` un `1` (misma empresa) o un `0` (distinta) por fila; lo que no sabe lo
deja vacio y no cuenta. Aqui, como la base trae la verdad (`COL_ENTIDAD`), la
etiqueta se **simula**: `1` si los dos registros pertenecen a la misma entidad.
Con datos propios ponga `COL_ENTIDAD = None` y deje el archivo etiquetado en
`RUTA_DATOS / ARCHIVO_ETIQUETAS`.

In [ ]:
if COL_ENTIDAD:
    entidad = base.set_index(COL_ID)[COL_ENTIDAD]
    pares["IS_MATCH"] = (pares["ID_1"].map(entidad) == pares["ID_2"].map(entidad)).astype(int)
    pares["REVIEWER"] = "simulado desde " + COL_ENTIDAD
else:
    ruta_etiquetas = RUTA_DATOS / ARCHIVO_ETIQUETAS
    if not ruta_etiquetas.exists():
        raise FileNotFoundError(
            f"No existe {ruta_etiquetas}: etiquete pares_para_etiquetar.xlsx (columna IS_MATCH, "
            f"1 o 0), guardelo como {ARCHIVO_ETIQUETAS} en {RUTA_DATOS} y vuelva a correr."
        )
    pares = pd.read_excel(ruta_etiquetas, dtype={"ID_1": str, "ID_2": str})
    if pares["IS_MATCH"].isna().all():
        raise ValueError(f"{ARCHIVO_ETIQUETAS} no tiene ningun IS_MATCH lleno: no hay verdad.")

VERDAD = pares.loc[pares["IS_MATCH"].notna()]
generador.guardar(pares, RUTA_RESULTADOS / "pares_etiquetados.csv")
# La regla de que es una etiqueta valida (1/0, "1"/"0", True/False) vive en la
# libreria: un "si" falla aqui con el mensaje que dice que corregir.
positivos = validar_etiquetas(VERDAD["IS_MATCH"])
print(f"{len(VERDAD)} pares etiquetados: {int(positivos.sum())} positivos, "
      f"{int((~positivos).sum())} negativos")
pd.crosstab(VERDAD["CASE_TYPE"], positivos.astype(int), colnames=["IS_MATCH"])

### 4 · Optimizar

`OrchestratorOptimizer` corre el motor completo (L1…L5) una vez por trial con
los parametros que Optuna propone dentro de `ESPACIO`, y mide cada corrida
**solo sobre los pares etiquetados** (`evaluation.pairwise.evaluar_pares_etiquetados`):
un par positivo acierta si el motor pone a sus dos registros en el mismo grupo, uno
negativo falla si los junta. Antes de optimizar se mide el perfil tal cual
(`evaluar_config({})`), que es la linea base contra la que se compara.

La semilla del muestreador es fija (`seed=42`): la misma base y el mismo espacio
dan el mismo estudio.

In [ ]:
optimizer = OrchestratorOptimizer(
    base_config=crear_config_orchestrator(perfil=PERFIL, validate=False),
    sources=FUENTES,
    truth=VERDAD,
    search_space=espacio_de_busqueda(ESPACIO),
    col_id=COL_ID,  # la columna con la que la correlativa se cruza con la verdad
)
antes = optimizer.evaluar_config({}, OBJETIVO)
if "error" in antes:  # la libreria dice que paso y que hacer; no seguir a ciegas
    raise ValueError(antes["error"])
print(f"Linea base ({PERFIL} sin ajustes): {OBJETIVO} = {antes['score']:.4f}  "
      f"[precision {antes['precision']:.3f} · recall {antes['recall']:.3f} · "
      f"{antes['n_pares_evaluados']} pares evaluados]")

resultado = optimizer.optimize(
    n_trials=N_TRIALS, optimization_target=OBJETIVO, show_progress_bar=False, seed=SEMILLA
)
print(f"Mejor {OBJETIVO}: {resultado['best_score']:.4f} en {resultado['total_trials_completed']} "
      f"trials completados · mejores parametros: {resultado['best_params']}")
historial = optimizer.history_df()
historial[[c for c in historial.columns if not c.startswith("metric_") or c in ("metric_precision", "metric_recall")]]

Dos figuras ligeras (matplotlib, sin dashboard): la **convergencia** —el score de
cada trial y el mejor acumulado— y la **importancia** de cada parametro segun
Optuna. Si la linea del mejor acumulado se aplana pronto, mas trials no van a
cambiar la respuesta; si un parametro pesa casi cero, no vale la pena afinarlo.

In [ ]:
graficar_convergencia(historial, objetivo="score", ruta=RUTA_RESULTADOS / "convergencia.png")
graficar_importancia(resultado["study"], ruta=RUTA_RESULTADOS / "importancia.png")
display(Image(filename=RUTA_RESULTADOS / "convergencia.png"))
display(Image(filename=RUTA_RESULTADOS / "importancia.png"))

### 5 · Aplicar y comparar

`optimizer.ajustes_perfil()` traduce los mejores parametros al formato que
`linkage(ajustes_perfil=...)` recibe: la misma regla con la que el optimizador armo
cada trial, escrita una sola vez. `linkage` deja la **carpeta del estandar de
salida** en `RUTA_RESULTADOS` y su correlativa se mide con
`optimizer.evaluar_correlativa` sobre los mismos pares: el score en produccion debe
ser el del estudio.

In [ ]:
AJUSTES = optimizer.ajustes_perfil()
print("ajustes_perfil:", AJUSTES)
res = linkage(
    FUENTES,
    col_nit=COL_NIT,
    col_name=COL_NOMBRE,
    col_ciudad=COL_CIUDAD,
    col_id=COL_ID,
    profile=PERFIL,
    ajustes_perfil=AJUSTES,
    work_dir=str(RUTA_TRABAJO / "optimizado"),
    carpeta_salida=RUTA_RESULTADOS,
    nombre="optimizado",
    skip_reporting=True,
)
# La correlativa del estandar conserva las columnas de la fuente: COL_ID cruza con la verdad.
despues_score, despues = optimizer.evaluar_correlativa(res.correlativa[[COL_ID, "ID_GRUPO"]], OBJETIVO)
comparacion = pd.DataFrame(
    {
        "linea base (sin ajustes)": antes,
        "mejor trial del estudio": resultado["best_metrics"],
        "linkage con ajustes": despues,
    }
).loc[["precision", "recall", "f1", "tp", "fp", "fn", "n_pares_evaluados"]]
print(f"{OBJETIVO}: {antes['score']:.4f} antes → {despues_score:.4f} despues "
      f"(estudio: {resultado['best_score']:.4f})")
print(f"Carpeta del estandar: {res.manifiesto.get('carpeta_salida', RUTA_RESULTADOS)}")
comparacion

### 6 · Guardar el estudio

`guardar_estudio` escribe un JSON con la version de la libreria, el objetivo, los
mejores parametros, los `ajustes_perfil` listos para `linkage()` y el historial de
cada trial (parametros, score, metricas, tiempo). Es lo que hace falta para
reproducir la decision y para aplicarla en otro notebook sin volver a optimizar.

In [ ]:
ruta_estudio = optimizer.guardar_estudio(RUTA_RESULTADOS / "estudio_optuna.json")
print(f"Estudio guardado en {ruta_estudio}")
print(sorted(p.name for p in RUTA_RESULTADOS.iterdir()))

### Como leer el resultado

- **`comparacion`**: si `linkage con ajustes` no reproduce el `mejor trial del
  estudio`, algo cambio entre el estudio y la aplicacion (otra base, otro perfil,
  otra version): no aplique los ajustes.
- **Pocos pares, poca confianza**: con 160 pares una diferencia de 0,02 en F1 son
  tres pares. Antes de llevar los ajustes a produccion, etiquete mas pares de la
  clase donde el motor falla (`CASE_TYPE` de los FP/FN) y vuelva a correr.
- **Lo que NO hace este notebook**: no inventa la verdad (la pone una persona), no
  optimiza el perfil global de la libreria (los valores por defecto se deciden con
  el banco institucional, `docs/BANCO.md`) y no mide `macro_f1` (exige una verdad
  por registro con `ESTRATO`). Ver [`docs/OPTIMIZACION.md`](../docs/OPTIMIZACION.md).
- **Para aplicar en otro notebook**: copie `ajustes_perfil` del JSON a
  `linkage(..., ajustes_perfil={...})`. Una clave desconocida falla de inmediato;
  nada se ignora en silencio.